# 02. Baseline PhoBERT (Phase 3)

Fine-tune `vinai/phobert-base`, phân 4 lớp. Một lần Save Version chỉ chạy một model:

| `RUN` | Plan | Input | Config |
|---|---|---|---|
| `3.1` | Model 3.1 | caption đã preprocess | `configs/text_only.yaml` |
| `3.2` | Model 3.2 | caption là feature chính; OCR và mô tả VLM là hai feature phụ có trọng số riêng | `configs/text_only_imgtext.yaml` |

Kaggle: bật **Internet** và **GPU**. Add Input:

1. `phuclkz/vimmsd-private-dataset` — ảnh và nhãn gốc
2. `vtnmai/vimmsd-data-preprocessed` — ba file `text_preprocessed.json`, `image_preprocessed.json`, `image_preprocessed.csv`

Train đọc hai file JSON. CSV là bảng để xem (cùng nội dung OCR/mô tả), không đưa vào model.

Output trong `/kaggle/working/outputs/<tên config>/`: `best.pt`, `test_metrics.json` (precision, recall, F1), `confusion_matrix_test.png`, `results_row.md` (thời gian train và peak VRAM).

In [ ]:
# === Cell bootstrap (giống nhau ở mọi notebook, xem CODEBASE.md mục 6) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/meo225/Multimodal_Sarcasm_Detection.git"
REF = "main"               # branch đã có loader text_preprocessed.json / image_preprocessed.json
CODE_DIR = "/tmp/IE403"    # clone ra ngoài /kaggle/working để token không bị lưu vào output


def get_github_token():
    """Token cho repo private; repo public thì không cần tạo secret."""
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ and not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):  # ảnh Kaggle dựng từ ảnh Colab nên cũng có biến này
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")  # kernel đang chạy chưa thấy package vừa cài -e
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # local: chạy từ gốc repo (đã `pip install -e .` một lần)

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

In [ ]:
import json
import logging
import shutil
from pathlib import Path

from IPython.display import Image as IPImage, display

from vimmsd.utils.config import load_config

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)

# Đổi thành "3.2" rồi Save Version một lần nữa cho model caption + OCR + VLM.
RUN = "3.1"
CONFIGS = {
    "3.1": "configs/text_only.yaml",
    "3.2": "configs/text_only_imgtext.yaml",
}
if RUN not in CONFIGS:
    raise ValueError(f"RUN phải là 3.1 hoặc 3.2, nhận được {RUN!r}")
CONFIG = CONFIGS[RUN]
print("sẽ train", RUN, CONFIG)

## Gắn data đã preprocess

`text_preprocessed.json` có dạng `{meta, texts}`. `texts` là caption gốc → caption đã tách từ. Model 3.1 lấy hết caption từ file này, không chạy lại VnCoreNLP.

`image_preprocessed.json` có dạng `{meta, images}`. Mỗi `image_key` (`train-images/...jpg`) có `ocr` và `description`. Model 3.2 encode caption, OCR và mô tả thành ba feature riêng. Caption hệ số 1, OCR hệ số 0.5, mô tả VLM hệ số 0.25. Câu OCR/VLM chưa nằm trong cache caption, nên lần chạy 3.2 vẫn tách từ phần chữ từ ảnh (cần Java).

In [ ]:
def find_preprocessed():
    root = Path("/kaggle/input")
    if not root.exists():
        local = Path("data/processed")
        if (local / "text_preprocessed.json").exists():
            return local
        raise FileNotFoundError(
            "Chạy trên Kaggle và Add Input vtnmai/vimmsd-data-preprocessed cùng phuclkz/vimmsd-private-dataset."
        )
    hits = sorted(root.rglob("text_preprocessed.json"))
    if not hits:
        raise FileNotFoundError(
            "Không thấy text_preprocessed.json trong /kaggle/input. Add Input dataset vtnmai/vimmsd-data-preprocessed."
        )
    return hits[0].parent


src = find_preprocessed()
required = ("text_preprocessed.json",) if RUN == "3.1" else ("text_preprocessed.json", "image_preprocessed.json")
for name in ("text_preprocessed.json", "image_preprocessed.json", "image_preprocessed.csv"):
    path = src / name
    if not path.exists():
        if name in required:
            raise FileNotFoundError(f"thiếu {path}")
        print(f"{name}: không có (không bắt buộc với model {RUN})")
        continue
    print(f"{name}: {path.stat().st_size / 1e6:.2f} MB")

cfg = load_config(CONFIG)
cache_dir = Path(cfg.paths.cache_dir)
cache_dir.mkdir(parents=True, exist_ok=True)
for name in ("text_preprocessed.json", "image_preprocessed.json"):
    if (src / name).exists():
        shutil.copy(src / name, cache_dir / name)

text_raw = json.loads((cache_dir / "text_preprocessed.json").read_text(encoding="utf-8"))
texts = text_raw["texts"]
print(f"caption trong cache: {len(texts)} | pipeline_key: {text_raw.get('meta', {}).get('pipeline_key')}")

if RUN == "3.2":
    if not shutil.which("java"):
        raise RuntimeError("Model 3.2 cần Java để tách từ phần OCR/VLM. Máy này không có java.")
    img_raw = json.loads((cache_dir / "image_preprocessed.json").read_text(encoding="utf-8"))
    images = img_raw["images"]
    sample_key, sample = next(iter(images.items()))
    if not isinstance(sample, dict) or "ocr" not in sample or "description" not in sample:
        raise ValueError(f"image_preprocessed.json không có ocr/description, mẫu {sample_key}: {sample!r}")
    print(f"ảnh trong cache: {len(images)} | missing: {img_raw.get('meta', {}).get('missing')}")
    print("mẫu", sample_key)

print("cache ghi được:", cache_dir)

## Train

Checkpoint `last.pt` lưu mỗi epoch. Session bị ngắt thì attach output version trước làm input và chạy lại với `--override train.resume_from=<đường dẫn>`.

In [ ]:
!python scripts/train.py --config {CONFIG}

## Precision, recall, F1

`test_metrics.json` là số cho task đánh giá. `results_row.md` có thời gian train và peak VRAM cho phần viết doc. Copy dòng đó vào `reports/results.md`.

Cột tên model Hugging Face trong plan điền sau khi upload `best.pt`. Notebook chưa đẩy checkpoint lên Hugging Face.

In [ ]:
out_dir = Path(cfg.paths.output_dir) / cfg.name
metrics = json.loads((out_dir / "test_metrics.json").read_text(encoding="utf-8"))
from vimmsd.training.metrics import format_metrics
print(format_metrics(metrics))
display(IPImage(str(out_dir / "confusion_matrix_test.png"), width=450))
print((out_dir / "results_row.md").read_text(encoding="utf-8"))

## Mẫu dự đoán sai trên test

In [ ]:
!python scripts/evaluate.py --config {CONFIG} --split test
import pandas as pd
pred = pd.read_csv(out_dir / "test_predictions.csv")
wrong = pred[pred.label != pred.pred]
print(f"{len(wrong)}/{len(pred)} mẫu sai")
wrong.groupby(["label", "pred"]).size().sort_values(ascending=False)

In [ ]:
wrong.sample(min(10, len(wrong)), random_state=0)[["id", "label", "pred", "caption"]]